# Multi-Input Document Image Classifier (PyTorch)

**Goal:** Demonstrate a CNN that uses both a 64×64 grayscale image and a document-type one-hot vector to classify a code *category*. This is **not full OCR**: the model does not transcribe text. All sample inputs below are **synthetic geometric patterns**, not insurance documents.

## 1. Setup
Run from this notebook folder or the project root. The next cell locates `src/` automatically. Install dependencies with `pip install -r requirements.txt`.

In [1]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
if not (root / "src").exists():
    root = root.parent
if not (root / "src").exists():
    raise FileNotFoundError("Run this notebook from the repository folder or notebooks/")
sys.path.insert(0, str(root))
import torch
import matplotlib.pyplot as plt
from src.ocr_model import OCRModel
from src.data import make_demo_dataset
from src.train import run_training
print("PyTorch:", torch.__version__)

PyTorch: 2.10.0+cpu


## 2. Inspect synthetic data
Categories are represented by different bright vertical bars. This is an intentionally simple code-pipeline demonstration, not a realistic scanned document task.

In [2]:
dataset = make_demo_dataset(n=160, seed=42)
image, onehot, label = dataset[0]
print("Image:", tuple(image.shape), "Type:", onehot.tolist(), "Label:", label.item())
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for j, ax in enumerate(axes):
    img, _, y = dataset[j]
    ax.imshow(img.squeeze(0), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Artificial class {y.item()}")
    ax.axis("off")
fig.suptitle("Synthetic geometric patterns (not real documents)")
fig.tight_layout()
plt.show()

Image: (1, 64, 64) Type: [0.0, 0.0, 0.0, 0.0, 1.0] Label: 0


/tmp/ipykernel_580/435969710.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Define and check architecture
`image_layer` has two 3×3, padding-1 convolutions and produces 128 features. `type_layer` produces 32 features. Concatenation feeds a four-class classifier.

In [3]:
model = OCRModel(num_types=5, num_classes=4)
print(model)
with torch.no_grad():
    logits = model(image.unsqueeze(0), onehot.unsqueeze(0))
assert logits.shape == (1, 4)
print("Logit tensor shape:", tuple(logits.shape))

OCRModel(
  (image_layer): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=8192, out_features=128, bias=True)
    (8): ReLU()
  )
  (type_layer): Sequential(
    (0): Linear(in_features=5, out_features=32, bias=True)
    (1): ReLU()
  )
  (classifier): Sequential(
    (0): Linear(in_features=160, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=4, bias=True)
  )
)
Logit tensor shape: (1, 4)


## 4. Train for 10 epochs
The learning curve and holdout accuracy are for artificially generated data only. Holdout splits in this tutorial are random, not grouped by customer/document.

In [4]:
model, history = run_training(
    dataset, output_dir=root / "outputs" / "notebook",
    epochs=10, batch_size=32, lr=0.001, seed=42
)
print("Final synthetic holdout accuracy:", round(history["val_accuracy"][-1], 3))

Epoch 01/10 | train loss 1.3513 | val loss 1.2270 | val accuracy 0.812
Epoch 02/10 | train loss 1.0830 | val loss 0.8247 | val accuracy 1.000


Epoch 03/10 | train loss 0.6161 | val loss 0.2997 | val accuracy 1.000
Epoch 04/10 | train loss 0.1809 | val loss 0.0417 | val accuracy 1.000


Epoch 05/10 | train loss 0.0207 | val loss 0.0026 | val accuracy 1.000
Epoch 06/10 | train loss 0.0011 | val loss 0.0002 | val accuracy 1.000


Epoch 07/10 | train loss 0.0001 | val loss 0.0000 | val accuracy 1.000
Epoch 08/10 | train loss 0.0000 | val loss 0.0000 | val accuracy 1.000


Epoch 09/10 | train loss 0.0000 | val loss 0.0000 | val accuracy 1.000
Epoch 10/10 | train loss 0.0000 | val loss 0.0000 | val accuracy 1.000


Final synthetic holdout accuracy: 1.0


## 5. Review training history

In [5]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, history["train_loss"], label="train loss")
axes[0].plot(epochs, history["val_loss"], label="val loss")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Training and validation loss")
axes[0].legend()
axes[1].plot(epochs, history["val_accuracy"], marker="o")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Synthetic validation accuracy", ylim=(0, 1.05))
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

/tmp/ipykernel_580/1531709403.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Inference on one synthetic example

In [6]:
model.eval()
img, kind, true_label = dataset[10]
with torch.no_grad():
    scores = model(img.unsqueeze(0), kind.unsqueeze(0))
    probs = scores.softmax(dim=1).squeeze().tolist()
print("Actual synthetic label:", true_label.item())
print("Predicted label:", scores.argmax(dim=1).item())
print("Category probabilities:", [round(p, 3) for p in probs])

Actual synthetic label: 2
Predicted label: 2
Category probabilities: [0.0, 0.0, 1.0, 0.0]


## 7. Next steps and limitations
- **Not OCR transcription:** text detection, code recognition, and CER/WER are outside this prototype.
- **Real evaluation:** use ethically sourced, redacted samples, grouped train/validation/test, F1 and confusion matrix.
- **Privacy:** do not commit personal identifiers, scanned policies, or private customer data.
- **Production:** test resolution sensitivity, skew, blur, class imbalance, generalization, security and human review.

The associated CLI is `python -m src.train --demo --epochs 10`; see `README.md`.